In [30]:
from pathlib import Path
import pandas as pd
import os
import numpy as np
from prototype import (
    VAE, train_vae, evaluate_latentgee,
    set_seed, load_hivrc, save_dated_filename, 
)

def load_hivrc(file_path):
    data_path = Path(f"{file_path}/insight.merged_otus.txt")
    meta_path = Path(f"{file_path}/SupplementaryMaterial.xlsx")
    
    dat = pd.read_csv(data_path, sep="\t", encoding = "utf-8")
    dat_meta = pd.read_excel(meta_path, header = 1, usecols = "B:L")

    dat_cols = dat_meta["SeqID"].astype(str).to_list()
    dat_cols.insert(0, 'Resphera Insight (Raw Counts)')
    dat_T = dat[dat_cols].T
    dat_T.reset_index(inplace = True)
    dat_T.columns = dat_T.iloc[0].astype(str)
    dat_T = dat_T.iloc[1:]
    dat_T.columns.values[0] = "SeqID"
    dat_merged = pd.merge(dat_T, dat_meta, on = "SeqID", how = "inner")
        
    X = dat_T.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
    # X_tensor = torch.tensor(X.values, dtype = torch.float32)
    
    return X, dat_merged
def compute_zero_proportion_by_prevalence(otu_df, cutoffs):
    n_samples = otu_df.shape[0]
    proportions = []

    for cutoff in cutoffs:
        # prevalence 계산: 각 OTU의 nonzero 비율
        prevalence = (otu_df > 0).sum(axis=0) / n_samples

        # cutoff 미만인 OTU 제거
        filtered_df = otu_df.loc[:, prevalence >= cutoff]

        # 전체 zero 비율 계산
        zero_count = (filtered_df == 0).sum().sum()
        total_count = filtered_df.size
        zero_proportion = zero_count / total_count

        proportions.append(zero_proportion)

    return proportions



In [ ]:
file_path = "/DATA/WGS_study/YSL/projects/Data"  
data_path = Path(f"{file_path}/insight.merged_otus.txt")
meta_path = Path(f"{file_path}/SupplementaryMaterial.xlsx")

dat = pd.read_csv(data_path, sep="\t", encoding = "utf-8")
dat_meta = pd.read_excel(meta_path, header = 1, usecols = "B:L")

dat_cols = dat_meta["SeqID"].astype(str).to_list()
dat_cols.insert(0, 'Resphera Insight (Raw Counts)')
dat_T = dat[dat_cols].T
dat_T.reset_index(inplace = True)
dat_T.columns = dat_T.iloc[0].astype(str)
dat_T = dat_T.iloc[1:]
dat_T.columns.values[0] = "SeqID"
dat_merged = pd.merge(dat_T, dat_meta, on = "SeqID", how = "inner")

X = dat_merged.iloc[1:5, 1:(dat_T.shape[1])].apply(pd.to_numeric, errors="coerce")
dat_cov = dat_merged[['hivstatus', 'Age', 'gender', 'msm','ARTuse', 'cd4']]



In [ ]:
print(dat_T.shape)
print(dat_merged.shape)
print(dat_meta.shape)
dat_cov = dat_merged[['hivstatus', 'Age', 'gender', 'msm','ARTuse', 'cd4']]





(1032, 38126)
(1032, 38136)
(1032, 11)


,hivstatus,Age,gender,msm,ARTuse,cd4
0,0,27.0,1.0,NaN,NaN,NaN
1,1,48.0,1.0,1.0,0.0,400.0
2,1,25.0,1.0,1.0,0.0,532.0
3,0,29.0,1.0,NaN,NaN,NaN
4,1,58.0,0.0,0.0,0.0,400.0
...,...,...,...,...,...,...
1027,0,42.0,1.0,1.0,NaN,NaN
1028,0,37.0,1.0,1.0,NaN,NaN
1029,0,54.0,1.0,1.0,NaN,NaN
1030,0,41.0,1.0,1.0,NaN,NaN


In [33]:
min_sample_ratio = 0.1
presence = (X_raw > 0).sum(axis=0)
prevalence = presence / X_raw.shape[0]
otu_df = X_raw.loc[:, prevalence > min_sample_ratio]

In [34]:
dat_merged[['Study', 'hivstatus', 'msm', 'Age', 'gender', 'cd4']]
# dat_merged.columns

,Study,hivstatus,msm,Age,gender,cd4
0,Dillon,0,NaN,27.0,1.0,NaN
1,Dillon,1,1.0,48.0,1.0,400.0
2,Dillon,1,1.0,25.0,1.0,532.0
3,Dillon,0,NaN,29.0,1.0,NaN
4,Dillon,1,0.0,58.0,0.0,400.0
...,...,...,...,...,...,...
1027,Yu,0,1.0,42.0,1.0,NaN
1028,Yu,0,1.0,37.0,1.0,NaN
1029,Yu,0,1.0,54.0,1.0,NaN
1030,Yu,0,1.0,41.0,1.0,NaN
